# multi_heatmap - 01 - Datos reales y 7 referencias

Notebook de **Google Colab** para la variante `universal_refs`:

* **7 referencias**: unipolar/Cz, linked mastoides, linked lobulos, bipolar (vecinos fisicos), CAR, REST y Laplaciano de superficie.
* **Cascos 100% reales**: subconjuntos exactos del canonico eegbci y bases BIDS externas con sus electrodos nativos (sin geometria simulada).
* Exploracion de la estructura temporal que motiva la cabeza dinamica.

**Uso:** Runtime -> GPU (T4) -> ejecutar en orden.

## 0 - Entorno

In [ ]:
# ---- 0 - Entorno (Colab o local) --------------------------------------
# Colab: Runtime > Change runtime type > GPU (T4). Rama: explore/multi-heatmap.
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/sonoAESS/universal-eeg-transformer.git"
BRANCH   = "explore/multi-heatmap"

IN_COLAB = "google.colab" in sys.modules or "/content" in os.getcwd()

if IN_COLAB:
    ROOT = Path("/content/universal-eeg-transformer")
    if not ROOT.exists():
        !git clone -b {BRANCH} {REPO_URL} {ROOT}
    %pip install -q mne pyyaml pandas matplotlib scikit-learn requests scipy
else:
    ROOT = Path.cwd()
    while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
print(f"ROOT = {ROOT}\nColab = {IN_COLAB}")

In [ ]:
# ---- 0b - Persistencia en Google Drive (opcional) ---------------------
USE_DRIVE = IN_COLAB

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/universal_eeg_cache")
    CACHE.mkdir(exist_ok=True)
    for link in ("data/processed", "runs"):
        target = CACHE / link.split("/")[-1]
        target.mkdir(parents=True, exist_ok=True)
        dest = ROOT / link
        if not dest.exists():
            dest.symlink_to(target)
    print("Cache y runs enlazados a:", CACHE)
else:
    print("Modo local: cache en ./data/processed y ./runs")

## 1 - Experimento

In [ ]:
# ---- 1 - Variante universal_refs ---------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

CONFIG = ROOT / "config/universal_refs.yaml"

from eeg_transform.nb import load_experiment
from eeg_transform.experiments.multi import multiconfig_summary
from eeg_transform.config import REFERENCE_KINDS

cfg, ds = load_experiment(CONFIG)
data = multiconfig_data(cfg, ds)

print(ds.summary())
print()
print(multiconfig_summary(data))
print(f"\nReferencias ({len(REFERENCE_KINDS)}x{len(REFERENCE_KINDS)} rutas por config):")
print(REFERENCE_KINDS)

## 2 - Geometria real

In [ ]:
# ---- 2 - Cascos reales --------------------------------------------------
# Solo distribuciones con electrodos REALES: subconjuntos exactos del
# canonico eegbci y bases externas BIDS (cuando esten ingestadas). Nada de
# geometrias simuladas.
fig, axes = plt.subplots(1, len(data.order), figsize=(4 * len(data.order), 4))
for ax, label in zip(np.atleast_1d(axes), data.order):
    mc = data[label]
    pos = mc.positions
    ax.scatter(pos[:, 0], pos[:, 1], c=pos[:, 2], s=14, cmap="viridis")
    ax.set_title(f"{label}\n({mc.n_channels} ch reales)")
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Distribuciones de electrodos (proyeccion XY, color = z)")
plt.tight_layout(); plt.show()

## 3 - Senales

In [ ]:
# ---- 3 - Las siete referencias (trazas, split test) --------------------
i0, n_show, ch = 100, 400, 10
refs_test = {k: ds.refs[k][ds.split_idx["test"]] for k in REFERENCE_KINDS}
names = list(ds.ch_names)

fig, axes = plt.subplots(len(REFERENCE_KINDS), 1, figsize=(12, 13), sharex=True)
for ax, k in zip(axes, REFERENCE_KINDS):
    sig = refs_test[k][i0 : i0 + n_show, ch]
    ax.plot(sig, lw=0.8)
    ax.set_ylabel(k); ax.grid(alpha=0.3)
axes[0].set_title(f"Canal '{names[ch]}' - 7 referencias (test)")
plt.xlabel("muestra"); plt.tight_layout(); plt.show()

print("RMS relativo por referencia (vs unipolar):",
      {k: round(float(np.std(refs_test[k]) / np.std(refs_test["unipolar"])), 3)
       for k in REFERENCE_KINDS})

## 4 - Operadores

In [ ]:
# ---- 4 - Matrices de los 7 operadores ---------------------------------
from eeg_transform.references import build_reference_matrix

label = "canonical"
mc = data[label]
ops = {
    k: build_reference_matrix(
        k, mc.n_channels,
        unipolar_ref_index=mc.unipolar_ref_index,
        lead_field=mc.leadfield,
        rest_rcond=cfg.leadfield.rest_rcond,
        positions=mc.positions,
    )
    for k in REFERENCE_KINDS
}

fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for ax, k in zip(axes.ravel(), REFERENCE_KINDS):
    im = ax.imshow(ops[k], cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(f"{k} (rango {np.linalg.matrix_rank(ops[k])})")
for ax in axes.ravel()[len(REFERENCE_KINDS):]:
    ax.axis("off")
fig.suptitle("Operadores de referencia (convencion X_ref = X @ M)")
plt.tight_layout(); plt.show()

## 5 - Potencial vs Laplaciano

In [ ]:
# ---- 5 - Campo de superficie: potencial vs laplaciano ------------------
grid_px = cfg.mapping.grid_px
split, i0 = "test", 42

fig, axes = plt.subplots(2, len(data.order),
                         figsize=(3.4 * len(data.order), 7))
for col, label in enumerate(data.order):
    mc = data[label]
    pot = (mc.refs[split]["unipolar"][i0] @ mc.surface.T)
    lap = (mc.refs[split]["laplacian"][i0] @ mc.surface.T)
    axes[0, col].imshow(pot.reshape(grid_px, grid_px), cmap="RdBu_r",
                        origin="lower")
    axes[1, col].imshow(lap.reshape(grid_px, grid_px), cmap="RdBu_r",
                        origin="lower")
    axes[0, col].set_title(label)
axes[0, 0].set_ylabel("unipolar"); axes[1, 0].set_ylabel("laplaciano")
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Potencial vs CSD en la malla compartida (misma muestra)")
plt.tight_layout(); plt.show()

## 6 - Dimension temporal

In [ ]:
# ---- 6 - Estructura temporal (justifica la ventana dinamica) -----------
from scipy.signal import welch

sfreq = 160.0
seg = refs_test["unipolar"][i0 : i0 + 4000]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
max_lag = 100
cols = range(0, seg.shape[1], max(1, seg.shape[1] // 6))
ac = np.stack([
    np.correlate(seg[:, c] - seg[:, c].mean(),
                 seg[: max_lag + 1, c] - seg[: max_lag + 1, c].mean()
                 )[::-1][: max_lag + 1] for c in cols])
axes[0].plot(np.arange(max_lag + 1) / sfreq * 1000,
             (ac / ac[:, :1]).T)
axes[0].set_xlabel("lag (ms)"); axes[0].set_title("Autocorrelacion (canales)")
axes[0].axvline(cfg.model.temporal_window / sfreq * 1000, color="r",
                ls="--", label=f"T_w = {cfg.model.temporal_window} muestras")
axes[0].legend(); axes[0].grid(alpha=0.3)

f, pxx = welch(seg, fs=sfreq, nperseg=256)
axes[1].semilogy(f, pxx.mean(axis=1))
axes[1].set_xlabel("Hz"); axes[1].set_title("PSD media")
for lo, hi, nm in ((1,4,"delta"),(8,13,"alpha"),(13,30,"beta")):
    axes[1].axvspan(lo, hi, alpha=0.12)
    axes[1].text((lo+hi)/2, pxx.mean(axis=1).max()*0.3, nm, ha="center")
plt.tight_layout(); plt.show()

print(f"Cabeza dinamica: ventana {cfg.model.temporal_window} "
      f"({cfg.model.temporal_window/sfreq*1000:.0f} ms), "
      f"stride {cfg.model.temporal_stride}")

## 7 - Base externa real

In [ ]:
# ---- 7 - Ingestar una base externa REAL (opcional) ---------------------
# 1) Censo de candidatos (donde la API de OpenNeuro sea accesible):
#      python tools/openneuro_census.py --probe
#      python tools/openneuro_census.py --output censo.csv
#
# 2) Descargar el dataset BIDS elegido (p. ej. denso EGI/GSN) e ingestarlo:
if False:   # <- activar con la ruta real descargada en Drive
    from eeg_transform.config import DataConfig, LeadFieldConfig
    from eeg_transform.data.external import build_external_dataset

    ext_ds, cache_file = build_external_dataset(
        bids_root="/content/drive/MyDrive/bids/<dataset>",
        label="<etiqueta>",            # -> config "external:<etiqueta>"
        cache_dir=cfg.dataset.cache_dir,
        data_cfg=DataConfig(bandpass=[1.0, 45.0]),
        leadfield_cfg=LeadFieldConfig(src_grid_mm=10.0),
    )
    print(cache_file)
    # anadir "external:<etiqueta>" a mapping.configs en el YAML y reconstruir.

## 8 - Balanceo

In [ ]:
# ---- 8 - Presupuesto balanceado ----------------------------------------
rows = []
for lbl in data.order:
    mc = data[lbl]
    row = {"config": lbl, "canales": mc.n_channels}
    row.update({s: mc.refs[s]["unipolar"].shape[0]
                for s in ("train", "val", "test")})
    rows.append(row)
pd.DataFrame(rows)

## Conclusiones de la exploracion

* Todas las distribuciones provienen de **electrodos reales**; las 7
  referencias son operadores lineales analiticos sobre cada casco.
* El **laplaciano** vive en otra escala espacial (alta frecuencia): mismo
  patron global que el potencial pero resaltando fuentes locales.
* La **memoria temporal** (autocorrelacion) y las oscilaciones por banda
  justifican la cabeza dinamica de ventana centrada: las rutas mal
  condicionadas (rest/laplaciano desde cascos escasos) son las que mas
  pueden beneficiarse.
* Siguiente paso: `02_modelo_entrenamiento.ipynb`.